### [Time Series Forecasting with XGBoost](https://levelup.gitconnected.com/time-series-forecasting-with-xgboost-dc5dc4510238)

The primary goal of this machine learning model is to accurately predict hourly energy consumption. By building a predictive model, we aim to forecast future energy demand based on historical patterns in the data. This can help utility providers, grid operators, and other stakeholders optimize resource allocation and manage supply more efficiently.

XGBoost is a popular machine learning algorithm known for its efficiency and performance, especially on structured or tabular data. Some reasons why XGBoost is well-suited for this project include:

- **High Accuracy:** _XGBoost_ is often one of the top-performing algorithms for regression and classification tasks.
- **Handling of Missing Data:** _XGBoost_ can handle missing values automatically, which is beneficial when working with real-world data.
- **Feature Engineering and Selection:** XGBoost can capture complex patterns in the data through gradient boosting, and it's able to identify important features automatically.
- **Fast Execution:** _XGBoost_ is optimized for speed, making it suitable for large datasets and computationally intensive tasks.

In [ ]:
import os

folder_to_delete = "/kaggle/input"

# Check if the folder exists before attempting to delete
if os.path.exists(folder_to_delete):
    !rm -rf {folder_to_delete}
    print(f"Folder '{folder_to_delete}' and its contents removed.")
else:
    print(f"Folder '{folder_to_delete}' does not exist.")

In [ ]:
%pip install -Uq optuna kagglehub

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Download latest version
import kagglehub
path = kagglehub.dataset_download("robikscube/hourly-energy-consumption", force_download=True)

print("Path to dataset files:", path)

In [ ]:
# Import libraries
import numpy as np            # For numerical operations and linear algebra, such as array and matrix manipulations
import pandas as pd           # For data processing, including data manipulation and reading/writing CSV files (e.g., pd.read_csv)
import optuna                 # For automated hyperparameter tuning, which optimizes model parameters to improve accuracy
import logging                # For controlling and managing log outputs, especially useful in suppressing or capturing log messages
import seaborn as sns         # For data visualization, particularly for creating aesthetically pleasing statistical graphics
import matplotlib.pyplot as plt  # For general-purpose plotting, allows for custom charts and visualizations

import xgboost as xgb         # XGBoost library, used for efficient gradient boosting, which is popular for structured/tabular data
from xgboost import plot_importance, plot_tree  # Additional XGBoost functions for visualizing feature importance and decision trees

from sklearn.metrics import mean_squared_error, mean_absolute_error  # Performance metrics for model evaluation, particularly useful for regression tasks

# Set color palette and style for visualizations
color_pal = sns.color_palette()    # Define a color palette for consistent styling across plots
plt.style.use('fivethirtyeight')   # Set plot style to 'fivethirtyeight' for a clean and professional look

import warnings
warnings.filterwarnings("ignore")  # Ignore warnings for cleaner output

In [ ]:
# Load dataset
df = pd.read_csv('/kaggle/input/hourly-energy-consumption/PJME_hourly.csv')
df = df.set_index("Datetime")
df.index = pd.to_datetime(df.index)

In [ ]:
# Load head data
df.head()

In [ ]:
# Load tail data
df.tail()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# Plotting the PJME energy use data with specific styling options
ax = df.plot(
    style=".",                    # Use dots to represent each data point on the plot
    figsize=(15, 5),              # Set the figure size to 15 inches wide by 5 inches tall
    color=color_pal[0],           # Use the first color from the color palette for the plot points
    title="PJME Energy Use in MW" # Set the title of the plot to indicate the data being displayed
)

# Add axis labels
ax.set_xlabel("Datetime")
ax.set_ylabel("Energy Consumption (MW)")

# Display the plot
plt.show()

In [ ]:
# Splitting the dataset into training and testing sets based on a date condition
# The training set includes data before January 1, 2015
# The testing set includes data from January 1, 2015, onwards
train = df.loc[df.index < '2015-01-01']
test = df.loc[df.index >= '2015-01-01']

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(15, 5))

# Plot training and test sets with customized colors
train.plot(ax=ax, label='Training Set', color='blue', linewidth=2)
test.plot(ax=ax, label='Test Set', color='orange', linewidth=2)

# Add vertical line for split date with annotation
split_date = '2015-01-01'
ax.axvline(split_date, color='black', linestyle='--', linewidth=1.5)
ax.text(split_date, ax.get_ylim()[1] * 0.9, 'Train/Test Split',
        rotation=90, color='black', verticalalignment='center', fontweight='bold')

# Customize title and labels
ax.set_title('Data Train/Test Split', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('Energy Consumption (MW)', fontsize=14)

# Display grid for better readability
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

# Add legend and show plot
ax.legend(['Training Set', 'Test Set'])
plt.show()

In [ ]:
# Filter the DataFrame based on the date range
filtered_df = df.loc[(df.index > "2010-01-01") & (df.index < "2010-08-01")]

# Apply a 7-day rolling average to smooth the data
smoothed_df = filtered_df.rolling(window=7, center=True).mean()

# Plot the smoothed data
ax = smoothed_df.plot(figsize=(15, 5), title="Energy Consumption from January to July 2010", color=color_pal[0])

# Add additional plot information for clarity
ax.set_xlabel("Date")  # X-axis label
ax.set_ylabel("Energy Consumption (MW)")  # Y-axis label
ax.grid(True)  # Enable grid for better readability

# Add legend if there are multiple columns
if len(smoothed_df.columns) > 1:
    ax.legend(title="Metrics")
plt.show()

In [ ]:
def create_features(df):
    # Existing features
    df = df.copy()
    df["hour"] = df.index.hour
    df["dayofweek"] = df.index.dayofweek
    df["quarter"] = df.index.quarter
    df["month"] = df.index.month
    df["year"] = df.index.year
    df["dayofyear"] = df.index.dayofyear

    # Additional features
    df["dayofmonth"] = df.index.day              # Day of the month (1 to 31)
    df["weekofyear"] = df.index.isocalendar().week  # Week of the year (1 to 52)
    df["is_weekend"] = df.index.dayofweek >= 5    # Binary feature for weekends (1 if weekend, else 0)
    df["is_month_start"] = df.index.is_month_start # Binary feature for start of month
    df["is_month_end"] = df.index.is_month_end     # Binary feature for end of month
    df["is_quarter_start"] = df.index.is_quarter_start # Binary feature for start of quarter
    df["is_quarter_end"] = df.index.is_quarter_end     # Binary feature for end of quarter
    df["is_year_start"] = df.index.is_year_start   # Binary feature for start of year
    df["is_year_end"] = df.index.is_year_end       # Binary feature for end of year

    # Cyclical features (useful for capturing seasonality patterns)
    df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24)    # Sine transformation for hour
    df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24)    # Cosine transformation for hour
    df["sin_dayofweek"] = np.sin(2 * np.pi * df["dayofweek"] / 7)  # Sine transformation for day of the week
    df["cos_dayofweek"] = np.cos(2 * np.pi * df["dayofweek"] / 7)  # Cosine transformation for day of the week
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12)  # Sine transformation for month
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12)  # Cosine transformation for month

    return df

df = create_features(df)

In [ ]:
df.head()

In [ ]:
# Set a larger figure size and style
fig, ax = plt.subplots(figsize=(15, 5))
sns.set_style("whitegrid")  # Set the background style to whitegrid for a cleaner look

In [ ]:
# Use a color palette for the boxplot
sns.boxplot(data=df, x="hour", y="PJME_MW", ax=ax, palette="coolwarm")  # Adjust color with a palette

# Add a title and style it
ax.set_title("PJME Energy Consumption by Hour of the Day", fontsize=18, fontweight="bold", color="darkblue")

# Customize the x and y axis labels
ax.set_xlabel("Hour of the Day", fontsize=14, color="darkblue")
ax.set_ylabel("Energy Consumption (MW)", fontsize=14, color="darkblue")

# Add gridlines for better readability
ax.grid(True, linestyle="--", alpha=0.6)

# Remove the top and right spines to give the plot a cleaner look
sns.despine(top=True, right=True)

# Show the plot
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))  # Use plt.subplots instead of plt.subplot
sns.boxplot(data=df, x="month", y="PJME_MW", ax=ax)  # Pass ax to sns.boxplot
ax.set_title("PJME Energy Consumption by Month")
ax.set_xlabel("Month")  # Optional: Label the x-axis
ax.set_ylabel("Energy Consumption (MW)")  # Optional: Label the y-axis
plt.show()

In [ ]:
# Apply feature engineering to train and test sets
train = create_features(train)
test = create_features(test)

In [ ]:
# Define features and target
TARGET = "PJME_MW"
FEATURES = [
    "hour", "dayofweek", "quarter", "month", "year", "dayofyear",
    "dayofmonth", "weekofyear", "is_weekend", "is_month_start",
    "is_month_end", "is_quarter_start", "is_quarter_end", "is_year_start",
    "is_year_end", "sin_hour", "cos_hour", "sin_dayofweek", "cos_dayofweek",
    "sin_month", "cos_month"
]
X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

In [ ]:
# Suppress Optuna's output by setting the logging level to WARNING
optuna.logging.set_verbosity(logging.WARNING)

In [ ]:
# Optuna objective function for hyperparameter tuning
def objective(trial):
    """
    Defines the objective function for Optuna hyperparameter tuning of the XGBRegressor model.
    Parameters:
    - trial: An Optuna Trial object which suggests values for the hyperparameters.
    Returns:
    - rmse: The Root Mean Squared Error (RMSE) on the test set, used as the objective to minimize.
    """

    # Define the hyperparameters with search spaces for Optuna to optimize
    param = {
        "n_estimators": 1000,  # Set a high number of estimators, allowing early stopping to determine optimal rounds
        "early_stopping_rounds": 50,  # Stops training if there's no improvement for 50 rounds
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.1, log=True),  # Controls step size, log scale
        "max_depth": trial.suggest_int("max_depth", 3, 10),  # Limits tree depth to control complexity
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),  # Minimum sum of weights in a child node
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),  # Fraction of data for each tree to avoid overfitting
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),  # Fraction of features per tree
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),  # Minimum loss reduction for split, log scale
        "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),  # L2 regularization, log scale
        "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),  # L1 regularization, log scale
        "eval_metric": "rmse"  # Evaluation metric set to RMSE (Root Mean Squared Error)
    }

    # Initialize the XGBRegressor model with the suggested hyperparameters
    model = xgb.XGBRegressor(**param)

    # Fit the model on the training set, with the validation set for early stopping
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],  # Use test set as a validation set for early stopping
        verbose=False  # Suppress output to keep the output clean
    )

    # Predict on the test set and calculate RMSE as the objective value
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))  # Calculate RMSE to evaluate model performance

    return rmse  # Return RMSE as the value to be minimized by Optuna

# Run Optuna optimization
study = optuna.create_study(direction="minimize")  # Create an Optuna study to minimize RMSE
study.optimize(objective, n_trials=50)  # Run 50 trials of hyperparameter optimization

# Train final model with best parameters
# Retrieve the best parameters found by Optuna and set additional fixed parameters
best_params = study.best_params
best_params["n_estimators"] = 1000
best_params["early_stopping_rounds"] = 50
best_params["eval_metric"] = "rmse"

# Initialize and train the final model using the best-found hyperparameters
model = xgb.XGBRegressor(**best_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],  # Use the test set as a validation set for early stopping
    verbose=False  # Suppress output for a clean log
)

# Evaluate the model
# Generate predictions on the test set and calculate the RMSE to assess performance
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("Test RMSE:", rmse)  # Print the final RMSE on the test set

# Plot feature importance
# Visualize the top 10 most important features to understand which features contributed most to the model's predictions
xgb.plot_importance(model, importance_type="weight", max_num_features=10)
plt.show()

In [ ]:
# Make predictions on the test set and add them as a new column in test
test['MW_Prediction'] = model.predict(X_test)  # Assuming 'model' is your trained model

In [ ]:
# Concatenate the test set with predictions and the training set
pjme_all = pd.concat([train, test], sort=False)

# Plot the actual vs. predicted energy consumption with improved styling
fig, ax = plt.subplots(figsize=(15, 5))

# Plot the actual values with a solid line
pjme_all['PJME_MW'].plot(ax=ax, color='#1f77b4', linewidth=2, label='Actual MW')

# Plot the predicted values with a dashed line for distinction
pjme_all['MW_Prediction'].plot(ax=ax, color='#ff7f0e', linestyle='--', linewidth=2, label='Predicted MW')

# Customize x and y labels
ax.set_xlabel("Date", fontsize=12, labelpad=10)
ax.set_ylabel("Energy Consumption (MW)", fontsize=12, labelpad=10)

# Set a title with larger font and padding
ax.set_title("Actual vs Predicted PJME Energy Consumption", fontsize=16, fontweight='bold', pad=20)

# Adjust the legend for clarity
ax.legend(loc="upper left", fontsize=10, frameon=True, title="Legend")

# Enable grid with light color for readability
ax.grid(True, linestyle='--', color='grey', alpha=0.5)

# Highlight a specific period, if needed (optional)
highlight_start, highlight_end = "2015-01-01", "2015-02-01"
ax.axvspan(highlight_start, highlight_end, color="black", alpha=0.1, label="Highlighted Period")

# Add annotations for peak or significant points (optional)
peak_date = pjme_all['PJME_MW'].idxmax()  # Find date of max energy usage
peak_value = pjme_all['PJME_MW'].max()    # Find max energy usage value
ax.annotate(
    f"Peak: {int(peak_value)} MW",
    xy=(peak_date, peak_value),
    xycoords="data",
    xytext=(peak_date, peak_value + 5000),  # Adjust annotation position
    arrowprops=dict(arrowstyle="->", color="red", lw=1.5),
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.3", edgecolor="red", facecolor="white", alpha=0.8)
)

# Show the plot
plt.show()

In [ ]:
# Plot the forecast with the actuals, both as points
fig, ax = plt.subplots(figsize=(15, 5))

# Plot actual values as orange points
pjme_all['PJME_MW'].plot(
    ax=ax,
    color='orange',        # Color for actual values (orange)
    linestyle='',          # No line connecting points
    marker='o',            # Use dots for actual values
    markersize=5,
    label='Actual MW'
)

# Plot predicted values as blue points
pjme_all['MW_Prediction'].plot(
    ax=ax,
    color='blue',          # Color for predicted values (blue)
    linestyle='',          # No line connecting points
    marker='o',            # Use dots for predicted values
    markersize=5,
    label='Predicted MW'
)

# Set x-axis limits to focus on January 2015
ax.set_xbound(lower='2015-01-01', upper='2015-02-01')

# Set y-axis limits for better scaling
ax.set_ylim(0, 60000)

# Add labels and title
ax.set_xlabel("Date", fontsize=12, labelpad=10)
ax.set_ylabel("Energy Consumption (MW)", fontsize=12, labelpad=10)
ax.set_title("January 2015 Forecast vs Actuals", fontsize=16, fontweight='bold', pad=20)

# Add a legend with custom titles
ax.legend(loc="upper left", frameon=True, fontsize=10)

# Enable grid for better readability
ax.grid(True, linestyle='--', alpha=0.6)

# Show the plot
plt.show()

In [ ]:
# Plot the forecast with the actuals as points
fig, ax = plt.subplots(figsize=(15, 5))  # Set figure size directly in subplots

# Plot actual values as red points
pjme_all['PJME_MW'].plot(
    ax=ax,
    color='red',               # Color for actual values (red)
    linestyle='',              # No line connecting points
    marker='o',                # Use dots for actual values
    markersize=6,              # Size of the markers for actual values
    label='Actual MW'
)

# Plot predicted values as blue points
pjme_all['MW_Prediction'].plot(
    ax=ax,
    color='blue',              # Color for predicted values (blue)
    linestyle='',              # No line connecting points
    marker='o',                # Use dots for predicted values
    markersize=6,              # Size of the markers for predicted values
    label='Predicted MW'
)

# Set x-axis and y-axis limits for focusing on the first week of January 2015
ax.set_xbound(lower='2015-01-01', upper='2015-01-08')
ax.set_ylim(0, 60000)

# Add labels and title with customized fonts and padding
ax.set_xlabel("Date", fontsize=12, labelpad=10)
ax.set_ylabel("Energy Consumption (MW)", fontsize=12, labelpad=10)
ax.set_title("First Week of January 2015: Forecast vs Actuals", fontsize=16, fontweight='bold', pad=20)

# Customize the legend for clarity
ax.legend(loc="upper left", frameon=True, fontsize=10)

# Enable grid lines for readability, with a lighter color and dashed style
ax.grid(True, linestyle='--', color='grey', alpha=0.5)

# Display the plot
plt.show()

In [ ]:
# Create the plot with specified figure size
fig, ax = plt.subplots(figsize=(15, 5))

# Plot actual values as orange points
pjme_all['PJME_MW'].plot(
    ax=ax,
    color='orange',             # Orange color for actual values
    linestyle='',               # No line connecting points for actual values
    marker='o',                 # Dots for actual values
    markersize=6,               # Size of markers
    label='Actual MW'
)

# Plot predicted values as blue points
pjme_all['MW_Prediction'].plot(
    ax=ax,
    color='blue',               # Blue color for predicted values
    linestyle='',               # No line connecting points for predicted values
    marker='o',                 # Dots for predicted values
    markersize=6,               # Size of markers
    label='Predicted MW'
)

# Set x-axis and y-axis limits to focus on the specified date range
ax.set_xbound(lower='2015-07-01', upper='2015-07-08')
ax.set_ylim(0, 60000)

# Add labels and title with customized fonts and padding
ax.set_xlabel("Date", fontsize=12, labelpad=10)
ax.set_ylabel("Energy Consumption (MW)", fontsize=12, labelpad=10)
ax.set_title("First Week of July 2015: Forecast vs Actuals", fontsize=16, fontweight='bold', pad=20)

# Customize the legend for clarity
ax.legend(loc="upper left", frameon=True, fontsize=10)

# Enable grid lines for readability, with a lighter color and dashed style
ax.grid(True, linestyle='--', color='grey', alpha=0.5)

# Display the plot
plt.show()

In [ ]:
# Function for Mean Absolute Percentage Error (MAPE)
def mean_absolute_percentage_error(y_true, y_pred):
    """Calculates MAPE given y_true and y_pred"""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
# Calculate error metrics on the test set
rmse = np.sqrt(mean_squared_error(y_true=test['PJME_MW'], y_pred=test['MW_Prediction']))
mae = mean_absolute_error(y_true=test['PJME_MW'], y_pred=test['MW_Prediction'])
mape = mean_absolute_percentage_error(y_true=test['PJME_MW'], y_pred=test['MW_Prediction'])
print(f"Our RMSE error is {rmse:.2f}")
print(f"Our MAE error is {mae:.2f}")
print(f"Our MAPE error is {mape:.2f}%")

In [ ]:
# Calculate error columns
test['error'] = test['PJME_MW'] - test['MW_Prediction']
test['abs_error'] = test['error'].abs()

In [ ]:
# Group errors by day and calculate mean errors
error_by_day = test.groupby(['year', 'month', 'dayofmonth']).mean()[['PJME_MW', 'MW_Prediction', 'error', 'abs_error']]

# Display top 10 over-forecasted days
over_forecasted_days = error_by_day.sort_values('error').head(10)
print("Top 10 Over-Forecasted Days:")
display(over_forecasted_days)

# Display top 10 worst absolute predicted days
worst_predicted_days = error_by_day.sort_values('abs_error', ascending=False).head(10)
print("\nTop 10 Worst Absolute Predicted Days:")
display(worst_predicted_days)

# Display top 10 best predicted days
best_predicted_days = error_by_day.sort_values('abs_error').head(10)
print("\nTop 10 Best Predicted Days:")
display(best_predicted_days)

In [ ]:
# Plot function for selected days with larger dots
def plot_selected_day(data, start_date, end_date, title):
    fig, ax = plt.subplots(figsize=(10, 5))

    # Plot predicted values as larger blue dots
    data['MW_Prediction'].plot(
        ax=ax,
        color='blue',
        linestyle='',  # No connecting lines
        marker='o',    # Dots for points
        markersize=10, # Larger markers
        label='Predicted MW'
    )

    # Plot actual values as larger orange dots
    data['PJME_MW'].plot(
        ax=ax,
        color='orange',
        linestyle='',  # No connecting lines
        marker='o',    # Dots for points
        markersize=10, # Larger markers
        label='Actual MW'
    )

    # Set axis limits and labels
    ax.set_ylim(0, 60000)
    ax.set_xbound(lower=start_date, upper=end_date)
    ax.set_title(title, fontsize=16)
    ax.set_xlabel("Date")
    ax.set_ylabel("Energy Consumption (MW)")
    ax.legend(loc="upper left", frameon=True, fontsize=10)

    # Enable grid for readability
    ax.grid(True, linestyle='--', color='grey', alpha=0.5)

    plt.show()

In [ ]:
# Plot worst predicted day (e.g., August 13, 2016)
plot_selected_day(pjme_all, '2016-08-13', '2016-08-14', "Aug 13, 2016 - Worst Predicted Day")

# Plot best predicted day (e.g., October 3, 2016)
plot_selected_day(pjme_all, '2016-10-03', '2016-10-04', "Oct 3, 2016 - Best Predicted Day")